# Stress, Anxiety, and Reward Learning (COVID-19) — Starter Notebook

427 adults from 47 countries, recruited on Prolific in April 2020 (the first COVID-19 wave), did two reward-learning tasks and filled in questionnaires on perceived stress (PSS), anxiety (STAI), depression (PHQ-9), and COVID-19 risk (Guitart-Masip, Walsh, Dayan & Olsson, 2023).

On each trial, participants chose one of three pictures. One of them was the "target", which paid off more often than the others.

* **Signalled task:** 5 games of 25 trials; the target pays off 75% of the time (others 25%), and participants are told when a new game starts.
* **Reversal task:** one game of about 125 trials; the target pays off 80% of the time (others 20%) and changes *without warning* every 20–30 trials.

**How this notebook is organized**

| Section | What it does |
|---|---|
| Settings | Where the data lives (the only cell you might edit) |
| Load Data | Imports libraries, loads and prepares the data |
| Inspect the Dataset | First rows, size, column types |
| Describe Variables | Summary statistics |
| Check Missing Data | What's missing and why it matters |
| Basic Visualizations | A few plots to get oriented |
| Variables You May Want to Explore | The main outcome and predictor variables |

This notebook is for **getting to know the dataset**. Every cell is starter code (🧰) written by the instructor; run them in order and read the output. You do your homework analysis in the separate homework notebooks, not here.

Open the table of contents (the ☰ icon on the left in Colab) to jump between sections.

## Settings

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

If the default link stops working, or you downloaded the file yourself, change the path below. See the dataset guide for where to get the data.

In [ ]:
# Settings: where the data lives.
# By default the data load from this course's GitHub repository, so there's nothing to download.
REPO = "cogs118d/datasets"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/data/06_stress_anxiety/"

# Cloned the repo and working on your own computer? Use the local folder instead:
# BASE_URL = "../data/06_stress_anxiety/"
# Saved the files to Google Drive? Point BASE_URL at that folder, e.g.:
# BASE_URL = "/content/drive/MyDrive/06_stress_anxiety/"
DATA_FOLDER = BASE_URL

# Load Data

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Libraries used in every COGS 118D starter notebook
import numpy as np                        # numbers and arrays
import pandas as pd                       # data tables ("DataFrames")
import matplotlib.pyplot as plt           # plots
import statsmodels.formula.api as smf     # GLMs and mixed models (you'll use this later)

pd.set_option("display.max_columns", 60)  # show more columns when printing tables
plt.rcParams["figure.figsize"] = (7, 4)   # default plot size

In [ ]:
reversal_raw = pd.read_csv(DATA_FOLDER + "reversal_n427.csv", low_memory=False)
signal_raw   = pd.read_csv(DATA_FOLDER + "signal_n427.csv", low_memory=False)
scores_raw   = pd.read_csv(DATA_FOLDER + "merge_all_factor_427.csv")
print("reversal:", reversal_raw.shape, "| signal:", signal_raw.shape, "| scores:", scores_raw.shape)

## Preprocessing

The raw files have about 100 columns, most of them questionnaire bookkeeping. We keep what matters for the tasks, put both tasks in one table, and attach one score per questionnaire.

In [ ]:
# The two task files name their response columns differently:
#   reversal task: Resp_3.corr (chose the target?) and Resp_3.rt (response time, seconds)
#   signalled task: Resp.corr and Resp.rt
def tidy_task(raw, task_name, correct_col, rt_col, key_col):
    t = pd.DataFrame({
        "participant": raw["Prolific.ID"],
        "task": task_name,
        "block": raw["gameCount"],             # game (signalled) or segment between target changes (reversal)
        "trial_in_block": raw["myCount"],      # trial number within the block
        "chose_target": raw[correct_col],      # 1 = chose the target picture
        "reward": raw["rewardGain"],           # 1 = got a reward on this trial
        "rt": raw[rt_col],                     # response time in seconds
        "responded": raw[key_col].notna().astype(int),
        "age": raw["Age.in.years"],
        "gender_raw": raw["Gender"],
        "education_raw": raw["Highest.level.of.education"],
        "residence_raw": raw["State.of.residence"],
    })
    # Trials with no key press are recorded as "not the target"; mark them missing instead
    t.loc[t["responded"] == 0, ["chose_target", "rt"]] = np.nan
    return t

rev = tidy_task(reversal_raw, "reversal", "Resp_3.corr", "Resp_3.rt", "Resp_3.keys")
sig = tidy_task(signal_raw, "signalled", "Resp.corr", "Resp.rt", "Resp.keys")
df = pd.concat([sig, rev], ignore_index=True)
df["trial_overall"] = df.groupby(["participant", "task"]).cumcount() + 1
print("No-response trials set to missing:", int((df["responded"] == 0).sum()))

In [ ]:
# Gender was typed in freely, so we clean it into three groups
g = df["gender_raw"].astype(str).str.strip().str.lower()
df["gender"] = np.select([g.isin(["female", "f", "feminine", "feale", "femalr"]),
                          g.isin(["male", "m", "masculine", "make"])], ["female", "male"], "other/unknown")

# One score per questionnaire (the authors' factor scores; higher = more of the trait)
score_cols = {"state_anxiety_congeneric": "state_anxiety", "trait_anxiety_congeneric": "trait_anxiety",
              "control_congeneric": "uncontrollable_stress", "self_efficacy_congeneric": "self_efficacy",
              "depression_congeneric_score": "depression", "covid_congeneric_score": "covid_risk"}
scores = scores_raw[["Prolific.ID"] + list(score_cols)].rename(columns={"Prolific.ID": "participant", **score_cols})
df = df.merge(scores, on="participant", how="left")

# What happened on the previous trial (within the same task), useful for win-stay / lose-shift questions
df["prev_chose_target"] = df.groupby(["participant", "task"])["chose_target"].shift(1)
df["prev_reward"] = df.groupby(["participant", "task"])["reward"].shift(1)

# One row per participant
people = df.groupby("participant").agg(age=("age", "first"), gender=("gender", "first")).reset_index().merge(scores, on="participant")
df[["participant", "task", "block", "trial_in_block", "chose_target", "reward", "rt", "state_anxiety"]].head()

# Inspect the Dataset

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# The first 5 rows: each row is one observation
df.head()

In [ ]:
# Dimensions: (number of rows, number of columns)
print("Rows, columns:", df.shape)

In [ ]:
# Column names and data types (numbers, text, dates, True/False)
df.info()

In [ ]:
print("Participants:", df["participant"].nunique())
print(df.groupby("task").size())

# Describe Variables

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
df.groupby("task")[["chose_target", "reward", "rt"]].mean().round(3)

In [ ]:
people[["age", "state_anxiety", "trait_anxiety", "uncontrollable_stress", "self_efficacy", "depression", "covid_risk"]].describe().round(2)

In [ ]:
people["gender"].value_counts()

# Check Missing Data

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Count and percent of missing values in each column (only columns with any missing)
missing = df.isna().sum()
missing_table = pd.DataFrame({"n_missing": missing, "percent": (100 * missing / len(df)).round(1)})
missing_table[missing_table["n_missing"] > 0].sort_values("n_missing", ascending=False)

**What to notice.** A few hundred trials had no response; they are now missing. The questionnaire scores are factor scores computed by the authors (standardized, so 0 is about average). Education and place of residence were typed in freely and are messy; clean them yourself if you want to use them.

# Basic Visualizations

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

These plots are for getting oriented. They are not answers to any homework question.

In [ ]:
# Learning within a block: proportion choosing the target by trial within block, for each task
for task_name in ["signalled", "reversal"]:
    t = df[(df["task"] == task_name) & (df["trial_in_block"] <= 25)]
    curve = t.groupby("trial_in_block")["chose_target"].mean()
    plt.plot(curve.index, curve.values, marker=".", label=task_name)
plt.axhline(1/3, color="gray", linestyle=":", label="chance (1 of 3)")
plt.xlabel("Trial within block")
plt.ylabel("Proportion choosing the target")
plt.legend()
plt.title("Learning curves")
plt.show()

In [ ]:
# Response times
plt.hist(df["rt"].dropna(), bins=60, edgecolor="white")
plt.xlabel("Response time (seconds)")
plt.ylabel("Number of trials")
plt.title("Response times")
plt.show()

In [ ]:
# Questionnaire scores
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, col in zip(axes, ["state_anxiety", "uncontrollable_stress", "depression"]):
    ax.hist(people[col], bins=30, edgecolor="white")
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Each participant's overall accuracy in each task
per_person = df.groupby(["participant", "task"])["chose_target"].mean().unstack()
plt.scatter(per_person["signalled"], per_person["reversal"], s=10, alpha=0.5)
plt.xlabel("Proportion target, signalled task")
plt.ylabel("Proportion target, reversal task")
plt.title("Same people, two tasks")
plt.show()

# Variables You May Want to Explore

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

**Outcome variables:** `chose_target` (yes/no), `rt`; per participant, overall accuracy or how often they switch away from the target after a loss (you'll need to compute this).

**Predictors:** `trial_in_block`, `task`, `prev_reward`, `prev_chose_target`, `state_anxiety`, `trait_anxiety`, `uncontrollable_stress`, `self_efficacy`, `depression`, `covid_risk`, `age`, `gender`.

In [ ]:
outcome_vars = ["chose_target", "rt"]
predictor_vars = ["trial_in_block", "prev_reward", "state_anxiety", "uncontrollable_stress", "depression"]
df[outcome_vars + predictor_vars].describe().round(2)